<a href="https://colab.research.google.com/github/carolmarquezini/preta_lab/blob/main/Projeto05_Analise_de_Dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Final de Análise de Dados com Python - Avaliação de Conhecimentos


## Introdução ao Projeto:

Neste projeto final, vocês irão utilizar o banco de dados da NBA (Basketball) disponível no Kaggle para realizar uma análise completa, incluindo ETL (Extração, Transformação e Carregamento de dados), manipulação de banco de dados, consulta e cruzamento em SQL, análise estatística, e a construção de insights a partir dos dados.

O objetivo é aplicar o que foi aprendido ao longo do curso sobre manipulação de arquivos e bancos de dados, análise de dados com Python, além de desenvolver um pensamento crítico e analítico para a tomada de decisões com base nos dados.


Os dados do Kaggle estão neste link: [NBA](https://www.kaggle.com/datasets/wyattowalsh/basketball)


## Resumo dos Dados:


O banco de dados contém diversas tabelas relacionadas à NBA, permitindo realizar uma análise detalhada sobre jogadores, times, estatísticas de jogos e muito mais. Abaixo, uma visão geral das tabelas e suas colunas principais:

1. Tabela 'Players':

Player_ID: Identificação única de cada jogador.

Player_Name: Nome do jogador.

Team: Equipe em que o jogador atua.

Position: Posição em que o jogador joga.

Height: Altura do jogador.

Birth_Date: Data de nascimento do jogador.


2. Tabela 'Teams':

Team_ID: Identificação única de cada time.

Team_Name: Nome da equipe.

City: Cidade em que o time está localizado.

Arena: Nome da arena onde a equipe joga seus jogos em
casa.

3. Tabela 'Games':

Game_ID: Identificação única de cada jogo.

Home_Team_ID: ID do time da casa.

Away_Team_ID: ID do time visitante.

Date: Data do jogo.

Home_Score: Pontuação do time da casa.

Away_Score: Pontuação do time visitante.

4. Tabela 'Game_Stats':

Game_ID: Identificação do jogo.

Player_ID: ID do jogador que participou do jogo.

Points: Pontos marcados pelo jogador.

Rebounds: Rebotes realizados pelo jogador.

Assists: Assistências realizadas pelo jogador

## O você precisa fazer/entrea:

In [95]:
# Montando o Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
%cd /content/drive/MyDrive/pretaLab-ciclo11/projeto05/

/content/drive/MyDrive/pretaLab-ciclo11/projeto05


In [78]:
!ls

Projeto05_Analise-de-Dados.ipynb


1. ETL - Construindo o Fluxo de Dados:

Manipulação de Banco de Dados: Conecte ao banco de dados SQLite sobre a NBA disponível no Kaggle. Verifique as tabelas existentes no banco e entenda a estrutura dos dados.

In [99]:
import sqlite3
import pandas as pd

In [80]:
# Conectando ao banco de dados (isso cria o arquivo se ele não existir)
connected=sqlite3.connect('nba.db')
cursor=connected.cursor()

In [81]:
# Criando a tabela Players
cursor.execute('''
CREATE TABLE IF NOT EXISTS Players (
    Player_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Player_Name TEXT NOT NULL,
    Team TEXT NOT NULL,
    Position TEXT,
    Height TEXT,
    Birth_Date TEXT
)
''')

# Criando a tabela Teams
cursor.execute('''
CREATE TABLE IF NOT EXISTS Teams (
    Team_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Team_Name TEXT NOT NULL,
    City TEXT NOT NULL,
    Arena TEXT
)
''')

# Criando a tabela Games
cursor.execute('''
CREATE TABLE IF NOT EXISTS Games (
    Game_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Home_Team_ID INTEGER,
    Away_Team_ID INTEGER,
    Date TEXT,
    Home_Score INTEGER,
    Away_Score INTEGER,
    FOREIGN KEY (Home_Team_ID) REFERENCES Teams (Team_ID),
    FOREIGN KEY (Away_Team_ID) REFERENCES Teams (Team_ID)
)
''')

# Criando a tabela Game_Stats
cursor.execute('''
CREATE TABLE IF NOT EXISTS Game_Stats (
    Game_ID INTEGER,
    Player_ID INTEGER,
    Points INTEGER,
    Rebounds INTEGER,
    Assists INTEGER,
    FOREIGN KEY (Game_ID) REFERENCES Games (Game_ID),
    FOREIGN KEY (Player_ID) REFERENCES Players (Player_ID),
    PRIMARY KEY (Game_ID, Player_ID)
)
''')

In [82]:
# Dados dos jogadores
players_data = [
    ('LeBron James', 'Los Angeles Lakers', 'Small Forward', '6\'9"', '1984-12-30'),
    ('Stephen Curry', 'Golden State Warriors', 'Point Guard', '6\'3"', '1988-03-14'),
    ('Kevin Durant', 'Brooklyn Nets', 'Small Forward', '6\'10"', '1988-09-29')
]

In [83]:
# Inserindo dados na tabela Players
try:
    cursor.executemany('INSERT INTO Players (Player_Name, Team, Position, Height, Birth_Date) VALUES (?, ?, ?, ?, ?)', players_data)
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
except sqlite3.OperationalError as e:
    print(f"OperationalError: {e}")

# Commitando as alterações
connected.commit()

In [84]:
# Dados das equipes
teams_data = [
    ('Los Angeles Lakers', 'Los Angeles', 'Staples Center'),
    ('Golden State Warriors', 'San Francisco', 'Chase Center'),
    ('Brooklyn Nets', 'Brooklyn', 'Barclays Center')
]

In [85]:
# Inserindo dados na tabela Teams
try:
    cursor.executemany('INSERT INTO Teams (Team_Name, City, Arena) VALUES (?, ?, ?)', teams_data)
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
except sqlite3.OperationalError as e:
    print(f"OperationalError: {e}")

# Commitando as alterações
connected.commit()

In [86]:
# Dados dos jogos
games_data = [
    (1, 2, '2023-10-01', 105, 95),  # Los Angeles Lakers vs Golden State Warriors
    (3, 1, '2023-10-02', 112, 110)  # Brooklyn Nets vs Los Angeles Lakers
]

In [87]:
# Inserindo dados na tabela Games
try:
    cursor.executemany('INSERT INTO Games (Home_Team_ID, Away_Team_ID, Date, Home_Score, Away_Score) VALUES (?, ?, ?, ?, ?)', games_data)
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
except sqlite3.OperationalError as e:
    print(f"OperationalError: {e}")

# Commitando as alterações
connected.commit()

In [88]:
# Dados das estatísticas dos jogos
game_stats_data = [
    (1, 1, 30, 10, 8),  # LeBron James - Jogo 1
    (1, 2, 25, 5, 6),   # Stephen Curry - Jogo 1
    (2, 1, 29, 11, 10), # LeBron James - Jogo 2
    (2, 3, 27, 7, 7)    # Kevin Durant - Jogo 2
]

In [89]:
# Inserindo dados na tabela Game_Stats
try:
    cursor.executemany('INSERT INTO Game_Stats (Game_ID, Player_ID, Points, Rebounds, Assists) VALUES (?, ?, ?, ?, ?)', game_stats_data)
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
except sqlite3.OperationalError as e:
    print(f"OperationalError: {e}")

# Commitando as alterações
connected.commit()

2. Consulta e Cruzamentos em SQL:

Utilize SQL para realizar cruzamentos entre as tabelas. Relacione as tabelas Players, Teams, e Game_Stats para descobrir o desempenho de jogadores em jogos específicos e suas respectivas equipes.

In [ ]:
# Exibir as primeiras linhas de cada tabela
players = pd.read_sql_query("SELECT * FROM Players LIMIT 5;", connected)
teams = pd.read_sql_query("SELECT * FROM Teams LIMIT 5;", connected)
games = pd.read_sql_query("SELECT * FROM Games LIMIT 5;", connected)
game_stats = pd.read_sql_query("SELECT * FROM Game_Stats LIMIT 5;", connected)

print(players)
print(teams)
print(games)
print(game_stats)


In [100]:
from google.cloud import bigquery

In [101]:
client = bigquery.Client()

In [ ]:
# Escrever a consulta SQL
query = """
SELECT *
FROM `analisede-dados-da-nba.NBA_Data`
LIMIT 5
"""

try:
    # Executar a consulta
    results = client.query(query).result()

    # Converter os resultados para um DataFrame do Pandas
    df_nba_data = results.to_dataframe()

    # Exibir os resultados
    print(df_nba_data)

except Exception as e:
    print("Erro ao executar a consulta:", e)

In [ ]:
SELECT
    p.Player_Name,
    t.Team_Name,
    gs.Points,
    gs.Rebounds,
    gs.Assists
FROM
    Game_Stats gs
JOIN
    Players p ON gs.Player_ID = p.Player_ID
JOIN
    Teams t ON p.Team = t.Team_ID;

3. Extração para DataFrame Pandas:

Extraia os dados processados do SQL para um DataFrame em pandas e faça o tratamento necessário, como lidar com valores faltantes e formatação correta de colunas.

4. Limpeza e Tratamento:

Verifique os dados extraídos em busca de inconsistências, como valores ausentes, tipos de dados incorretos ou duplicatas. Realize o tratamento adequado para garantir uma análise precisa.

5. Análise Estatística e Visualizações:

- Manipulação dos Dados

Vamos explorar formas avançadas de manipular os dados usando apply, lambda, cut e groupby para extrair insights. Estas operações são fundamentais para um entendimento mais profundo das variáveis, bem como para a preparação de dados para análises estatísticas.

 1. Calcular a pontuação por jogador usando a função apply e uma função lambda:

 2. Agrupar os dados por time e calcular a média de pontuação, rebotes e assistências usando groupby:

 3. Classificar jogadores com base na pontuação usando cut e criar categorias de performance:


  - Estatísticas Descritivas

Após a manipulação inicial dos dados, os alunos devem calcular as principais estatísticas descritivas para variáveis de interesse, como pontuação, rebotes e assistências:

 1. Média, Mediana, Desvio Padrão e Quartis

- Probabilidade e Amostragem

Aqui, os alunos devem calcular a probabilidade de eventos específicos ocorrerem, como a probabilidade de um time ganhar ou um jogador alcançar uma pontuação acima de um determinado valor.

1. Probabilidade de um jogador marcar mais de 20 pontos;

2. Criar amostras de 30% dos dados para realizar análises estatísticas posteriores.

- Testes de Hipóteses

Agora, vocês devem realizar testes de hipóteses. Suponha que queremos testar se a média de pontos dos jogadores em casa é diferente da média de pontos dos jogadores em jogos fora de casa. Para isso, utilizaremos o Teste T.

6. Visualizações dos Dados

As visualizações são uma parte crucial da análise de dados, pois ajudam a apresentar os resultados de forma clara e compreensível.

1. Boxplot da Pontuação por Categoria de Jogador
2. Distribuição de Idade dos Jogadores com histograma
3. Gráfico de Barras mostrando a Relação entre Pontuação e Rebotes por Time

7. Conclusão da Análise

Construa uma conclusão que resuma os achados na análise.